In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams["font.family"] = ["PingFang HK", "STHeiti", "Arial Unicode MS", "sans-serif"]

In [ ]:
OUTPUTS = Path("/Users/fe-edward/wafer-map-detection/outputs")

CLASSES = ["Center", "Donut", "Edge-Loc", "Edge-Ring", "Loc", "Near-full", "Random", "Scratch", "none"]

def load_per_class_f1(run_dir):
    meta = json.loads((OUTPUTS / run_dir / "metadata.json").read_text())
    idx_to_class = {v: k for k, v in meta["class_to_idx"].items()}
    return {idx_to_class[i]: f1 for i, f1 in enumerate(meta["per_class_f1"])}

# Read per-class F1 from metadata.json for the first three experiments
baseline_32 = load_per_class_f1("baseline_20260525_204323")
class_wt    = load_per_class_f1("base_loss_20260527_223003")
baseline_64 = load_per_class_f1("baseline_20260530_211352")

# Two-Stage: no combined metadata.json exists; values from the final test report
two_stage = {
    "Center":    0.9291,
    "Donut":     0.9405,
    "Edge-Loc":  0.8407,
    "Edge-Ring": 0.9848,
    "Loc":       0.8065,
    "Near-full": 0.9744,
    "Random":    0.9482,
    "Scratch":   0.7959,
    "none":      0.9862,
}

experiments = {
    "Baseline 32²": baseline_32,
    "Class Wt.": class_wt,
    "Baseline 64²": baseline_64,
    "Two-Stage": two_stage,
}

In [ ]:
# Print the data table for verification
header = f"{'類別':<12}" + "".join(f"{name:>14}" for name in experiments)
print(header)
print("-" * len(header))
for cls in CLASSES:
    row = f"{cls:<12}" + "".join(f"{exp[cls]:>14.4f}" for exp in experiments.values())
    print(row)

In [ ]:
n_classes = len(CLASSES)
n_models  = len(experiments)
bar_width = 0.18
x = np.arange(n_classes)

colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]

fig, ax = plt.subplots(figsize=(14, 6))

for i, (model_name, f1_dict) in enumerate(experiments.items()):
    f1_values = [f1_dict[cls] for cls in CLASSES]
    offset = (i - (n_models - 1) / 2) * bar_width
    bars = ax.bar(x + offset, f1_values, bar_width, label=model_name,
                  color=colors[i], edgecolor="white", linewidth=0.5)

ax.set_xlabel("缺陷類別", fontsize=13)
ax.set_ylabel("F1 Score", fontsize=13)
ax.set_title("各類別 F1 跨實驗比較", fontsize=15, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(CLASSES, fontsize=11)
ax.set_ylim(0, 1.08)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1f}"))
ax.axhline(y=1.0, color="grey", linewidth=0.8, linestyle="--", alpha=0.5)
ax.legend(fontsize=11, loc="lower right")
ax.grid(axis="y", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUTS / "f1_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to outputs/f1_comparison.png")